# 04 - Video Inference, BoT-SORT Audit & Tracker Baselines (`SORT`, `ByteTrack`) (WORKING COPY - TARGETED FIX)
**Master Workspace:** `MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026/08_COLAB_NOTEBOOKS`  
**Original Notebook Preserved:** `04_Video_Inference_Tracking.ipynb`  

### Targeted Fixes & Algorithmic Disclosures Applied in This Notebook:
1. **Fixed `OfflineTrackerReplayer` Invocation (`BUG-03`):** Corrected constructor call `OfflineTrackerReplayer(method=t_name)` and method call `run_on_cached_detections(det_df, enable_experiment=ENABLE_TRACKER_REPLAY)` (plus backward-compatible alias `run_tracker()`).
2. **Fixed DataFrame Positional Indexing (`BUG-04`):** Added `.reset_index(drop=True)` after `.sort_values(["frame", "confidence"], ascending=[True, False], kind="mergesort")` so `grp.index` strictly matches `0..N-1` indices in `out_tids`.
3. **Fixed ByteTrack Velocity Updates (`BUG-05`):** Added exponential moving average velocity updates (`0.5 * vel + 0.5 * (new_bbox - old_bbox)`) and velocity-predicted boxes across both Stage 1 (high-confidence) and Stage 2 (low-confidence) association in `bytetrack`.
4. **Connected End-to-End Tracker -> Fixed-Seat Identity -> AHP-SAW Pipeline (`BUG-06`):** Added `evaluate_tracker_pipeline_end_to_end()` to pass `botsort_cached`, `sort`, and `bytetrack` trajectories directly through `src/identity.py` (`assign_track_to_seat`, `validate_seat_assignment`, `build_student_level_detections`), 25 FPS duration matrix aggregation, and `AHP-SAW` rank correlation comparison.
5. **Explicit Algorithmic Fidelity Disclosure:** Clarifies in code and output tables that `OfflineTrackerReplayer` (`sort` and `bytetrack` modes) is a **Simplified Constant-Velocity EMA + Hungarian IoU Offline Replay Baseline** on cached detections (not an 8-state Kalman filter implementation), while native Ultralytics `bytetrack.yaml` / `botsort.yaml` full-Kalman video inference is provided via `run_native_ultralytics_tracker()` (locked behind `ENABLE_FULL_VIDEO_INFERENCE = False`).


In [1]:
# ==============================================================================
# CENTRALIZED GOOGLE DRIVE PATHS, EXECUTION LOCKS & MULTI-DETECTOR SELECTOR
# ==============================================================================
import json
import logging
import os
import subprocess
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Select Active Detector Model for Video Inference & Tracking:
#   "yolo11m"      -> Primary YOLO11m baseline (uses cached 06_TRACKING_AND_IDENTITY_VALIDATION/detections.csv)
#   "yolov8m"      -> YOLOv8m baseline (writes to 09_REVALIDATED_EXPERIMENT_OUTPUTS/multi_detector_pipeline/yolov8m/)
#   "rfdetr_small" -> RF-DETR-Small ("S", writes to 09_REVALIDATED_EXPERIMENT_OUTPUTS/multi_detector_pipeline/rfdetr_small/)
#   "rfdetr_large" -> RF-DETR-Large ("L", writes to 09_REVALIDATED_EXPERIMENT_OUTPUTS/multi_detector_pipeline/rfdetr_large/)
#   "ALL_TRAINED"  -> Sequentially processes all detectors whose trained checkpoints exist
ACTIVE_DETECTOR_MODEL = "ALL_TRAINED"
ENABLE_FULL_VIDEO_INFERENCE = True
ENABLE_TRACKER_REPLAY = True

ENABLE_ROBOFLOW_STAGING_DOWNLOAD = False
ENABLE_DATASET_REPLACEMENT = False
ENABLE_TRAINING = False
ENABLE_MODEL_EVALUATION = False
ENABLE_MCDM_EVALUATION = False
ENABLE_IDENTITY_ABLATION = False
ENABLE_SEAT_PERTURBATION_SIMULATION = False
ENABLE_TEMPORAL_EXECUTION = False
ENABLE_MONTE_CARLO = False
ENABLE_GPU_EXPERIMENTS = False

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s")
logger = logging.getLogger("YOLO11_IJITCS_SyncedColab")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

MASTER_ROOT = Path(
    "/content/drive/MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026"
)
DATASET_ROOT = MASTER_ROOT / "03_DATASET_AUDIT" / "YOLO11_DATASET_UNZIPPED"
DATA_YAML_PATH = DATASET_ROOT / "data_colab.yaml"
REPO_ROOT = MASTER_ROOT / "05_GITHUB_CODE_AND_MODULES" / "YOLO11_Repository"
MODULES_DIR = MASTER_ROOT / "05_GITHUB_CODE_AND_MODULES" / "Reconstructed_Research_Modules"
VIDEO_PATH = (
    MASTER_ROOT
    / "02_ORIGINAL_RESEARCH_ARCHIVE"
    / "Original_Videos"
    / "video_part001.mp4"
)
CHECKPOINT_DIR = MASTER_ROOT / "04_MODEL_AND_CONFIGURATION" / "Checkpoints"
OUTPUT_DIR = MASTER_ROOT / "09_REVALIDATED_EXPERIMENT_OUTPUTS"
OUTPUT_PATH = OUTPUT_DIR
MULTI_DETECTOR_PIPELINE_DIR = OUTPUT_DIR / "multi_detector_pipeline"

# Register REPO_ROOT and MODULES_DIR on sys.path
for _p in (REPO_ROOT, MODULES_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
importlib.invalidate_caches()
for _m in list(sys.modules.keys()):
    if _m == "src" or _m.startswith("src."):
        del sys.modules[_m]

if IN_COLAB:
    for _pkg_import, _pip_spec in (
        ("roboflow", "roboflow"),
        ("ultralytics", "ultralytics"),
        ("supervision", "supervision"),
    ):
        try:
            __import__(_pkg_import)
        except ModuleNotFoundError:
            logger.info("Installing missing Colab dependency: %s ...", _pip_spec)
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pip_spec])

from src.paths_config import get_detector_pipeline_paths
from src.detector_adapters_v2_aligned import (
    EXPECTED_CLASS_MAPPING,
    run_multi_detector_video_pipeline,
)

def _resolve_existing(candidates: list[Path]) -> Path:
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]

TRACKING_VAL_DIR = MASTER_ROOT / "06_TRACKING_AND_IDENTITY_VALIDATION"
AHP_SAW_VAL_DIR = MASTER_ROOT / "07_AHP_SAW_AND_STATISTICAL_VALIDATION"
STUDENT_SEATS_PATH = _resolve_existing([
    REPO_ROOT / "data" / "student_seats.csv",
    TRACKING_VAL_DIR / "student_seats.csv",
])

_active_paths = get_detector_pipeline_paths(ACTIVE_DETECTOR_MODEL, master_root=MASTER_ROOT)
DETECTIONS_CSV_PATH = _active_paths["detections_csv"]
TRACK_SUMMARY_CSV_PATH = _active_paths["track_summary_csv"]
TRACK_TO_STUDENT_CSV_PATH = _active_paths["track_to_student_csv"]
STUDENT_AUDIT_CSV_PATH = _active_paths["student_audit_csv"]
IDENTITY_ASSIGNMENTS_CSV_PATH = _active_paths["identity_assignments_csv"]
DURATION_MATRIX_CSV_PATH = _active_paths["duration_matrix_csv"]

logger.info("Active Detector in Notebook 04: %s | Detections CSV: %s", ACTIVE_DETECTOR_MODEL, DETECTIONS_CSV_PATH)

Mounted at /content/drive


## Step 2: Tracker Replay Adapters (`BoT-SORT`, Simplified `SORT-EMA-IoU`, Simplified `ByteTrack-EMA-IoU`) & End-to-End Identity/AHP-SAW Pipeline


In [2]:
# ==============================================================================
# MULTI-DETECTOR VIDEO INFERENCE, TRACKER REPLAY ADAPTERS (`BoT-SORT`, `SORT-EMA-IoU`,
# `ByteTrack-EMA-IoU`) & END-TO-END FIXED-SEAT IDENTITY + AHP-SAW EVALUATION
# ==============================================================================
import src.identity as base_identity_module
from src.aggregate import build_activity_matrix, build_both_representations
from src.ahp import ahp_weights
from src.detect_track import run_detection_tracking, summarise_tracks
from src.mcdm_topsis_electre_adapters import evaluate_all_mcdm_methods
from src.paths_config import get_detector_pipeline_paths
from src.saw import saw_score, zero_diagnostics
from src.tracker_adapters_aligned import (
    OfflineTrackerReplayer,
    evaluate_tracker_pipeline_end_to_end,
    run_native_ultralytics_tracker,
)

if str(ACTIVE_DETECTOR_MODEL).strip().upper() == "ALL_TRAINED":
    _detector_list = ["yolo11m", "yolov8m", "rfdetr_small", "rfdetr_large"]
else:
    _detector_list = [ACTIVE_DETECTOR_MODEL]

if ENABLE_FULL_VIDEO_INFERENCE or ENABLE_TRACKER_REPLAY:
    seats_df = base_identity_module.load_seat_map(str(STUDENT_SEATS_PATH), expected_students=21)
    for _det_key in _detector_list:
        _p = get_detector_pipeline_paths(_det_key, master_root=MASTER_ROOT)
        _norm_key = _p["detector_key"]
        print(f"\n================ DETECTOR: {_norm_key} ================")
        if _norm_key != "yolo11m" and (ENABLE_FULL_VIDEO_INFERENCE or not _p["detections_csv"].exists() or not _p["duration_matrix_csv"].exists()):
            if not _p["checkpoint_path"].exists():
                logger.warning("Skipping %s because trained checkpoint %s does not exist yet.", _norm_key, _p["checkpoint_path"])
                continue
            pipe_res = run_multi_detector_video_pipeline(
                detector_key=_norm_key,
                master_root=MASTER_ROOT,
                video_path=VIDEO_PATH,
                seats_csv_path=STUDENT_SEATS_PATH,
                base_identity_module=base_identity_module,
                tracker_replayer_cls=OfflineTrackerReplayer,
                conf_thresh=0.25,
                fps=25.0,
                total_video_frames=23471,
                enable_video_inference=ENABLE_FULL_VIDEO_INFERENCE,
            )
            print("Video Pipeline Summary:", json.dumps(pipe_res, indent=2))

        if ENABLE_TRACKER_REPLAY:
            if not _p["detections_csv"].exists():
                raise FileNotFoundError(
                    f"Missing {_p['detections_csv']}. For YOLOv8m / RF-DETR, set ENABLE_FULL_VIDEO_INFERENCE = True first."
                )
            det_df = pd.read_csv(_p["detections_csv"])
            trk_out_dir = (
                OUTPUT_DIR / "tracker_benchmarks"
                if _norm_key == "yolo11m"
                else _p["pipeline_dir"] / "tracker_benchmarks"
            )
            tracker_summary_df, tracker_rankings = evaluate_tracker_pipeline_end_to_end(
                detections_df=det_df,
                seats_df=seats_df,
                base_identity_module=base_identity_module,
                mcdm_eval_fn=evaluate_all_mcdm_methods,
                output_dir=trk_out_dir,
                fps=25.0,
                total_video_frames=23471,
                enable_experiment=ENABLE_TRACKER_REPLAY,
            )
            tracker_summary_df.insert(0, "detector_model", _norm_key)
            tracker_summary_df.to_csv(trk_out_dir / f"tracker_baseline_end_to_end_comparison_{_norm_key}.csv", index=False)
            print(tracker_summary_df.to_string(index=False))
else:
    logger.info(
        "[LOCKED] Multi-detector video inference & tracker replay (ACTIVE_DETECTOR_MODEL = '%s') "
        "are locked (ENABLE_FULL_VIDEO_INFERENCE = False, ENABLE_TRACKER_REPLAY = False).",
        ACTIVE_DETECTOR_MODEL,
    )


================ DETECTOR: yolo11m ================
detector_model tracker_method                                                                                                                            algorithmic_scope  n_tracked_detections  n_unique_tracks  n_primary_assigned  primary_assignment_rate_pct  n_assigned_students  mean_preprop_seat_purity_pct  mean_student_frame_coverage_pct  n_long_gap_fragments  ahp_saw_spearman_rho_vs_botsort  ahp_saw_spearman_p_value  ahp_saw_kendall_tau_vs_botsort  ahp_saw_mean_rank_shift  ahp_saw_top5_overlap_ratio
       yolo11m botsort_cached                           Exact Ultralytics BoT-SORT (with_reid=False, gmc=sparseOptFlow, Kalman filter + IoU) cached from video_part001.mp4                497848             1399              484093                        97.24                   21                         98.77                            98.22                    12                           1.0000              0.000000e+00               

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[2026-11-16 11:46:45] [WARNING] rf-detr - Checkpoint has 4 classes but model is configured for 90. Using checkpoint class count (4). Pass num_classes=4 to suppress this warning.
[2026-11-16 11:46:45] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. Create predictor = native_model.inference(...) and use predictor.predict(...). For a checkpoint, first load native_model = RFDETR.from_checkpoint(...).


Video Pipeline Summary: {
  "detector_key": "rfdetr_small",
  "pipeline_dir": "/content/drive/MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026/09_REVALIDATED_EXPERIMENT_OUTPUTS/multi_detector_pipeline/rfdetr_small",
  "total_detections": 512578,
  "total_tracks": 292,
  "identity_validation": {
    "expected_students": 21,
    "n_assigned_students": 21,
    "assigned_student_ids": [
      "S01",
      "S02",
      "S03",
      "S04",
      "S05",
      "S06",
      "S07",
      "S08",
      "S09",
      "S10",
      "S11",
      "S12",
      "S13",
      "S14",
      "S15",
      "S16",
      "S17",
      "S18",
      "S19",
      "S20",
      "S21"
    ],
    "n_unassigned_detections": 3783,
    "n_ambiguous_detections": 0,
    "n_duplicate_detections": 21093,
    "n_fragmented_students": 20,
    "n_unique_track_ids": 292,
    "n_track_fragments_merged": 201,
    "n_raw_detections": 512578,
    "n_primary_assigned": 487702,
    "primary_assignment_rate": 0.9514688496189848,
    "duplic

[2026-11-16 11:57:25] [WARNING] rf-detr - Checkpoint has 4 classes but model is configured for 90. Using checkpoint class count (4). Pass num_classes=4 to suppress this warning.
[2026-11-16 11:57:25] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. Create predictor = native_model.inference(...) and use predictor.predict(...). For a checkpoint, first load native_model = RFDETR.from_checkpoint(...).


Video Pipeline Summary: {
  "detector_key": "rfdetr_large",
  "pipeline_dir": "/content/drive/MyDrive/YOLO11_AHP_SAW_IJITCS_REVALIDATION_2026/09_REVALIDATED_EXPERIMENT_OUTPUTS/multi_detector_pipeline/rfdetr_large",
  "total_detections": 519753,
  "total_tracks": 290,
  "identity_validation": {
    "expected_students": 21,
    "n_assigned_students": 21,
    "assigned_student_ids": [
      "S01",
      "S02",
      "S03",
      "S04",
      "S05",
      "S06",
      "S07",
      "S08",
      "S09",
      "S10",
      "S11",
      "S12",
      "S13",
      "S14",
      "S15",
      "S16",
      "S17",
      "S18",
      "S19",
      "S20",
      "S21"
    ],
    "n_unassigned_detections": 7662,
    "n_ambiguous_detections": 0,
    "n_duplicate_detections": 25384,
    "n_fragmented_students": 21,
    "n_unique_track_ids": 290,
    "n_track_fragments_merged": 190,
    "n_raw_detections": 519753,
    "n_primary_assigned": 486707,
    "primary_assignment_rate": 0.9364197994047172,
    "duplic